# Retrain Model 2 (conditions) on t5-small with bigger, cleaner ORD data (Kaggle GPU)

The chemistry-pretrained ReactionT5-base attempt for Model 2 was abandoned: its
SMILES-only tokenizer can't represent the JSON/English targets, and the vocab-extension
fix, while it removed the `<unk>` corruption, failed to converge in Kaggle's training
environment across three configurations (RESULTS.md). t5-small (generic C4 pretraining)
already handles these targets natively -- no `<unk>`, no vocab surgery -- so the productive
lever for Model 2 is **more and cleaner data**, not a different base model.

**What changed vs the current t5-small Model 2:** trained on the 300k-pool conditions
split instead of the 60k-pool one. `data/v2_ord_train_300k/conditions_train.jsonl` has
**138,869** rows (3.4x the current 41,139), is **leak-free** (0% of its 7,714-row
`conditions_test` products appear in train -- verified locally) and **fully deduplicated**
(0 exact (product, reactants) duplicates). Same target format, same t5-small base, same
recipe otherwise (`lr=5e-4`, the script's tuned default for plain t5-small).

**Run this as Save & Run All (Commit).** Turn on **Internet** and **GPU accelerator**.

**Data:** `kuzmenkooleh/retro-planner-ord-conditions-300k` (uploaded from
`data/v2_ord_train_300k/conditions_{train,val,test}.jsonl`).

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print("  Device", i, torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

**Input data.** Resolve the dataset path robustly (Kaggle mounts vary).

In [ ]:
import os, glob
dataset_slug = "retro-planner-ord-conditions-300k"  # @param {type:"string"}
candidates = [f"/kaggle/input/{dataset_slug}", f"/kaggle/input/datasets/kuzmenkooleh/{dataset_slug}"]
base = next((c for c in candidates if os.path.exists(os.path.join(c, "conditions_train.jsonl"))), None)
if base is None:
    found = glob.glob("/kaggle/input/**/conditions_train.jsonl", recursive=True)
    listing = os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "MISSING"
    assert found, f"conditions_train.jsonl not found under /kaggle/input. Contents: {listing}"
    base = os.path.dirname(found[0])
train_file = os.path.join(base, "conditions_train.jsonl")
val_file = os.path.join(base, "conditions_val.jsonl")
print("base:", base)
print("train:", sum(1 for _ in open(train_file)), "rows |", "val:", sum(1 for _ in open(val_file)), "rows")

In [ ]:
output_dir = "/kaggle/working/model2_conditions_t5small_300k"  # @param {type:"string"}
time_budget_minutes = 170  # @param {type:"number"}
base_model = "t5-small"  # @param {type:"string"}
# t5-small (~60M) is tiny; 4 epochs over 138k rows on a single T4 fits comfortably.
# lr=5e-4 is the script's default, tuned for plain (non-chemical) t5-small.

In [ ]:
import os
os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"
!python scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --learning-rate 5e-4 \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_work \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print(f"Done (or paused at time budget). Log: {log_path}")

Log is redirected to `train.log` (avoids a huge notebook DOM on long runs). Check progress via `kaggle kernels status <user>/<slug>` or the Output tab.

**When done:** `output_dir/final` is the model. Evaluate on the held-out 300k test set (7,714 rows, leak-free) -- and, for a like-for-like comparison, evaluate the *existing* 60k-pool t5-small on the **same** test set:

```
python scripts/evaluate_conditions_model_topk.py \
    --test-file /kaggle/input/retro-planner-ord-conditions-300k/conditions_test.jsonl \
    --model-dir <downloaded_final_dir> \
    --num-beams 10 --device cuda --batch-size 32 \
    --output experiments/v2_model2_topk/conditions_t5small_300k_topk.json
```